<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/06-complex-nested-data/00-arrays.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)

print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))


# 6.0 — Arrays: explode and higher-order functions

The setup cell defines `EVENTS_SCHEMA` — the nested contract for
`events.jsonl` that all Module 6 notebooks share. (DROPMALFORMED skips
the corrupt line 10.)

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, IntegerType,
    DoubleType, ArrayType,
)

spark = (
    SparkSession.builder
    .appName("6.0-arrays")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

EVENTS_SCHEMA = StructType([
    StructField("event_id", LongType(), False),
    StructField("ts",       StringType(), True),
    StructField("action",   StringType(), True),
    StructField("user", StructType([
        StructField("id",      StringType(), True),
        StructField("country", StringType(), True),
    ]), True),
    StructField("items", ArrayType(StructType([
        StructField("sku", StringType(),  True),
        StructField("qty", IntegerType(), True),
    ])), True),
    StructField("payment", StructType([
        StructField("method", StringType(), True),
        StructField("amount", DoubleType(), True),
    ]), True),
])

events = (
    spark.read.schema(EVENTS_SCHEMA)
    .option("mode", "DROPMALFORMED")
    .json(f"{DATA_DIR}/events.jsonl")
)
events.printSchema()
print("events:", events.count(), "(12 lines - 1 corrupt)")

## Array basics — and the size() = -1 gotcha

In [ ]:
events.select(
    "event_id", "action",
    F.size("items").alias("n_items"),          # -1 = null array, 0 = empty!
    F.element_at("items", 1).alias("first_item"),   # 1-indexed
).show(truncate=False)

## explode vs explode_outer — watch events 4 and 6

In [ ]:
plain = events.select("event_id", F.explode("items").alias("item"))
outer = events.select("event_id", F.explode_outer("items").alias("item"))

print("events:", events.count(), "| explode:", plain.count(), "| explode_outer:", outer.count())
outer.filter(col("item").isNull()).show()   # the rows plain explode silently dropped

In [ ]:
# posexplode: element index comes along
events.filter(F.size("items") > 1) \
    .select("event_id", F.posexplode("items").alias("pos", "item")) \
    .select("event_id", "pos", "item.sku", "item.qty").show()

## Higher-order functions: compute WITHOUT changing shape

In [ ]:
events.select(
    "event_id",
    F.aggregate("items", F.lit(0), lambda acc, it: acc + it["qty"]).alias("total_qty"),
    F.transform("items", lambda it: it["sku"]).alias("skus"),
    F.filter("items", lambda it: it["qty"] >= 2).alias("bulk_items"),
    F.exists("items", lambda it: it["qty"] >= 3).alias("has_bulk"),
).show(truncate=False)

In [ ]:
# Proof the lambda is an expression, not a UDF: it appears IN THE PLAN
events.select(F.transform("items", lambda it: it["qty"] * 2).alias("double_qty")).explain()

## The anti-pattern vs the HOF, side by side

In [ ]:
# Same answer two ways: total qty per event
via_hof = events.select(
    "event_id",
    F.aggregate("items", F.lit(0), lambda a, it: a + it["qty"]).alias("total"),
)

via_explode = (
    events.select("event_id", F.explode_outer("items").alias("item"))
    .groupBy("event_id")
    .agg(F.coalesce(F.sum("item.qty"), F.lit(0)).alias("total"))
)

print(via_hof.orderBy("event_id").collect() == via_explode.orderBy("event_id").collect())
via_explode.explain()   # note the Exchange (shuffle) the HOF version doesn't have

## When explode IS right: elements as relational rows (join, cross-event agg)

In [ ]:
products = spark.createDataFrame(
    [("A1", 9.99), ("B2", 18.50), ("C3", 6.75), ("D4", 74.50), ("E5", 4.20)],
    "sku STRING, price DOUBLE",
)

sku_stats = (
    events.select(F.explode("items").alias("item"))
    .select("item.sku", "item.qty")
    .join(products, "sku")
    .groupBy("sku")
    .agg(F.sum("qty").alias("units"), F.round(F.sum(col("qty") * col("price")), 2).alias("revenue"))
)
sku_stats.orderBy(col("revenue").desc()).show()

## Exercises

1. Per event: the sku with the LARGEST qty — using only HOFs (`array_sort` with a comparator, or `F.aggregate` carrying a struct accumulator). No explode.
2. `arrays_zip`: build two literal arrays (`skus`, `qtys`) with `F.transform`, zip them back together, and confirm you reconstructed `items`.
3. Reassembly round trip: explode items, join prices, rebuild `items_priced` as `array<struct<sku,qty,price>>` per event with `collect_list(struct(...))`. Check the plan for the shuffle you paid.
4. The dedup twist: some pipelines have duplicate skus per event. Using `F.filter` + `F.array_distinct` + `F.transform`, dedupe `items` by sku IN PLACE, keeping the first occurrence's qty. (Hard — sketch is fine.)

In [ ]:
# your exercise code here
